In [2]:
import numpy as np
import nbimporter
from matplotlib import pyplot as plt
from scipy.optimize import curve_fit
from cdt.data import load_dataset
from scipy.stats import gamma, norm
from sklearn.metrics import roc_auc_score
import os
import random
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from cdt.causality import pairwise
from sklearn.preprocessing import MinMaxScaler
import cepairsimplementation as ce
seedR = random.Random(42)
seedN = np.random.default_rng()
from sklearn.gaussian_process import GaussianProcessRegressor

import torch
import torch as th
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

No GPU automatically detected. Setting SETTINGS.GPU to 0, and SETTINGS.NJOBS to cpu_count.


In [3]:
class MyNetwork(nn.Module):
    def __init__(self, d1, d2, h):
        """
        Initializes the network.
        
        Parameters:
        - d1: Dimension of the first input (E1) and the first output (X).
        - d2: Dimension of the second input (E2) and the second output (Y).
        - h:  Dimension of the hidden layers.
        """
        # Always call the parent constructor when inheriting from nn.Module.
        super(MyNetwork, self).__init__()
        
        # First stage: From E1 to X.
        # fc1: fully connected layer mapping E1 (d1) to a hidden representation (h).
        self.fc1 = nn.Linear(d1, h)
        # fc2: fully connected layer mapping the hidden representation to X (d1 outputs).
        self.fc2 = nn.Linear(h, d1)
        
        # Second stage: From concatenated [X, E2] to Y.
        # fc3: fully connected layer mapping the concatenated vector of dimension (d1+d2) to hidden size h.
        self.fc3 = nn.Linear(d1 + d2, h)
        # fc4: fully connected layer mapping the hidden representation to Y (d2 outputs).
        self.fc4 = nn.Linear(h, d2)

    def forward(self, E1, E2):
        """
        Forward pass through the network.
        
        Parameters:
        - E1: Tensor of shape (batch_size, d1)
        - E2: Tensor of shape (batch_size, d2)
        
        Returns:
        - X: Intermediate output from the first stage (batch_size, d1)
        - Y: Final output from the network (batch_size, d2)
        """
        # Pass E1 through the first linear layer and apply ReLU activation.
        hidden1 = F.relu(self.fc1(E1))
        # Compute X. (Note: we do not apply an activation after fc2, but you could if needed.)
        X = self.fc2(hidden1)
        
        # Concatenate X with E2 along the feature dimension.
        # torch.cat takes a tuple of tensors and a dimension along which to concatenate.
        concat = torch.cat((X, E2), dim=1)
        
        # Pass the concatenated tensor through another fully connected layer with ReLU activation.
        hidden2 = F.relu(self.fc3(concat))
        # Compute Y.
        Y = self.fc4(hidden2)
        
        return X, Y


In [4]:
class CdtNetwork(nn.Module):
    def __init__(self, d1, d2, h):
        """
        Initializes the network.
        
        Parameters:
        - d1: Dimension of the first input (E1) and the first output (X).
        - d2: Dimension of the second input (E2) and the second output (Y).
        - h:  Dimension of the hidden layers.
        """
        # Always call the parent constructor when inheriting from nn.Module.
        super(CdtNetwork, self).__init__()
        
        # Second stage: From concatenated [X, E2] to Y.
        # fc3: fully connected layer mapping the concatenated vector of dimension (d1+d2) to hidden size h.
        self.fc3 = nn.Linear(d1 + d2, h)
        # fc4: fully connected layer mapping the hidden representation to Y (d2 outputs).
        self.fc4 = nn.Linear(h, d2)

    def forward(self, X, E2):
        """
        Forward pass through the network.
        
        Parameters:
        - E1: Tensor of shape (batch_size, d1)
        - E2: Tensor of shape (batch_size, d2)
        
        Returns:
        - X: Intermediate output from the first stage (batch_size, d1)
        - Y: Final output from the network (batch_size, d2)
        """
        
        # Concatenate X with E2 along the feature dimension.
        # torch.cat takes a tuple of tensors and a dimension along which to concatenate.
        concat = torch.cat((X, E2), dim=1)
        
        # Pass the concatenated tensor through another fully connected layer with ReLU activation.
        hidden2 = F.relu(self.fc3(concat))
        # Compute Y.
        Y = self.fc4(hidden2)
        
        return X, Y

In [5]:
def gaussian_kernel(source, target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):
    """
    Computes a multi-kernel Gaussian (RBF) matrix between source and target.
    
    Parameters:
    - source: Tensor of shape (n, features)
    - target: Tensor of shape (m, features)
    - kernel_mul: A multiplier for bandwidth.
    - kernel_num: Number of different bandwidths to use.
    - fix_sigma: If provided, fixes the bandwidth.
    
    Returns:
    - Sum of kernel matrices computed using different bandwidths.
    """
    n_samples = int(source.size(0)) + int(target.size(0))
    # Concatenate source and target along the batch dimension.
    total = torch.cat([source, target], dim=0)
    
    # Expand the total tensor so we can compute pairwise distances.
    total0 = total.unsqueeze(0).expand(total.size(0), total.size(0), total.size(1))
    total1 = total.unsqueeze(1).expand(total.size(0), total.size(0), total.size(1))
    
    # Compute the L2 distance matrix.
    L2_distance = ((total0 - total1) ** 2).sum(2)
    
    # If fix_sigma is not given, compute the bandwidth as the mean distance.
    if fix_sigma:
        bandwidth = fix_sigma
    else:
        bandwidth = torch.sum(L2_distance.data) / (n_samples**2 - n_samples)
    
    # Adjust bandwidth using kernel_mul.
    bandwidth /= kernel_mul ** (kernel_num // 2)
    bandwidth_list = [bandwidth * (kernel_mul**i) for i in range(kernel_num)]
    
    # Compute the kernel matrices for each bandwidth and sum them.
    kernel_val = [torch.exp(-L2_distance / bandwidth_temp) for bandwidth_temp in bandwidth_list]
    return sum(kernel_val)

def mmd_loss(source, target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):
    """
    Computes the Maximum Mean Discrepancy (MMD) loss between two samples: source and target.
    
    MMD is defined as:
      MMD^2 = E[k(x,x')] + E[k(y,y')] - 2 E[k(x,y)]
    
    Parameters:
    - source: Tensor of shape (batch_size, features) for the source distribution.
    - target: Tensor of shape (batch_size, features) for the target distribution.
    - kernel_mul, kernel_num, fix_sigma: Parameters for the Gaussian kernel.
    
    Returns:
    - A scalar tensor representing the MMD loss.
    """
    batch_size = int(source.size(0))
    kernels = gaussian_kernel(source, target, kernel_mul, kernel_num, fix_sigma)
    
    # Split the combined kernel matrix into parts.
    XX = kernels[:batch_size, :batch_size]
    YY = kernels[batch_size:, batch_size:]
    XY = kernels[:batch_size, batch_size:]
    YX = kernels[batch_size:, :batch_size]
    
    # The MMD loss is the mean of these kernel components.
    loss = torch.mean(XX + YY - XY - YX)
    return loss

In [6]:
class MMDloss(th.nn.Module):
    """**[torch.nn.Module]** Maximum Mean Discrepancy Metric to compare
    empirical distributions.

    The MMD score is defined by:

    .. math::
        \\widehat{MMD_k}(\\mathcal{D}, \\widehat{\\mathcal{D}}) = 
        \\frac{1}{n^2} \\sum_{i, j = 1}^{n} k(x_i, x_j) + \\frac{1}{n^2}
        \\sum_{i, j = 1}^{n} k(\\hat{x}_i, \\hat{x}_j) - \\frac{2}{n^2} 
        \\sum_{i,j = 1}^n k(x_i, \\hat{x}_j)

    where :math:`\\mathcal{D} \\text{ and } \\widehat{\\mathcal{D}}` represent 
    respectively the observed and empirical distributions, :math:`k` represents
    the RBF kernel and :math:`n` the batch size.

    Args:
        input_size (int): Fixed batch size.
        bandwiths (list): List of bandwiths to take account of. Defaults at
            [0.01, 0.1, 1, 10, 100]
        device (str): PyTorch device on which the computation will be made.
            Defaults at ``cdt.SETTINGS.default_device``.

    Inputs: empirical, observed
        Forward pass: Takes both the true samples and the generated sample in any order 
        and returns the MMD score between the two empirical distributions.

        + **empirical** distribution of shape `(batch_size, features)`: torch.Tensor
          containing the empirical distribution
        + **observed** distribution of shape `(batch_size, features)`: torch.Tensor
          containing the observed distribution.

    Outputs: score
        + **score** of shape `(1)`: Torch.Tensor containing the loss value.

    .. note::
        Ref: Gretton, A., Borgwardt, K. M., Rasch, M. J., Schölkopf, 
        B., & Smola, A. (2012). A kernel two-sample test.
        Journal of Machine Learning Research, 13(Mar), 723-773.

    Example:
        >>> from cdt.utils.loss import MMDloss
        >>> import torch as th
        >>> x, y = th.randn(100,10), th.randn(100, 10)
        >>> mmd = MMDloss(100)  # 100 is the batch size
        >>> mmd(x, y)
        0.0766
    """

    def __init__(self, input_size, bandwidths=None):
        """Init the model."""
        super(MMDloss, self).__init__()
        if bandwidths is None:
            bandwidths = th.Tensor([0.01, 0.1, 1, 10, 100])
        else:
            bandwidths = bandwidths
        s = th.cat([th.ones([input_size, 1]) / input_size,
                    th.ones([input_size, 1]) / -input_size], 0)

        self.register_buffer('bandwidths', bandwidths.unsqueeze(0).unsqueeze(0))
        self.register_buffer('S', (s @ s.t()))

    def forward(self, x, y):
        X = th.cat([x, y], 0)
        # dot product between all combinations of rows in 'X'
        XX = X @ X.t()
        # dot product of rows with themselves
        # Old code : X2 = (X * X).sum(dim=1)
        # X2 = XX.diag().unsqueeze(0)
        X2 = (X * X).sum(dim=1).unsqueeze(0)
        # print(X2.shape)
        # exponent entries of the RBF kernel (without the sigma) for each
        # combination of the rows in 'X'
        exponent = -2*XX + X2.expand_as(XX) + X2.t().expand_as(XX)
        b = exponent.unsqueeze(2).expand(-1,-1, self.bandwidths.shape[2]) * -self.bandwidths
        lossMMD = th.sum(self.S.unsqueeze(2) * b.exp())
        return lossMMD

In [7]:
def cgnn_score(x,y,h=20,batch_size=128,num_epochs=10,plot=False):
    plot=False    # Infer feature dimensions.
    d1 = x.shape[1]
    d2 = y.shape[1]
    
    #pass to torch
    target_x = torch.from_numpy(x).float()
    target_y = torch.from_numpy(y).float()

    # Hyperparameters.
    h = 20           # Hidden layer dimension.
    learning_rate = 1e-2
    
    # Initialize model and optimizer.
    model = CdtNetwork(d1, d2, h)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Training loop over epochs.
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        
        # Optionally shuffle the dataset at each epoch.
        indices = torch.randperm(len(target_x))
        target_x = target_x[indices]
        target_y = target_y[indices]
        
        # Iterate over mini-batches.
        for i in range(0, len(x), batch_size):
            end_i = i + batch_size
            batch_target_x = target_x[i:end_i]
            batch_target_y = target_y[i:end_i]
            current_batch_size = batch_target_x.shape[0]
            
            # Sample E1 and E2 for the current batch.
            E2 = torch.randn(current_batch_size, d2)
            
            # Forward pass.
            pred_X, pred_Y = model(batch_target_x, E2)
            pred_concat = torch.cat([pred_X, pred_Y], dim=1)
            batch_target_concat = torch.cat([batch_target_x, batch_target_y], dim=1)
            
            # Compute the MMD loss.
            loss = mmd_loss(pred_concat, batch_target_concat)
            
            # Backpropagation.
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1
        
        avg_loss = epoch_loss / num_batches if num_batches > 0 else float('inf')
        #print(f"Epoch [{epoch+1}], Average MMD Loss: {avg_loss:.4f}")
    if plot==True:
        plt.plot(x,y,'.')
        for _ in range(30):
            E2_rand = torch.randn(1, d2)
            #    Expand it to match the number of points in x_lin_tensor.
            xs=np.linspace(min(x),max(x),1000)
            x_lin_tensor = torch.from_numpy(xs.reshape(-1, 1)).float()
            E2_expanded = E2_rand.expand(x_lin_tensor.shape[0], -1)
            model.eval()
            with torch.no_grad():
                _,ys=model(x_lin_tensor,E2_expanded)
            plt.plot(xs,ys.numpy()[:,0])
        plt.show()
        print(avg_loss)
    return avg_loss

def cgnn_score_tiago(x,y,h=20,batch_size=128,num_epochs=10):
        # Infer feature dimensions.
    d1 = x.shape[1]
    d2 = y.shape[1]
    
    #pass to torch
    target_x = torch.from_numpy(x).float()
    target_y = torch.from_numpy(y).float()

    # Hyperparameters.
    h = 20           # Hidden layer dimension.
    learning_rate = 1e-2  # For demonstration, using a single epoch.
    
    # Initialize model and optimizer.
    model = MyNetwork(d1, d2, h)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Training loop over epochs.
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        
        # Optionally shuffle the dataset at each epoch.
        indices = torch.randperm(len(target_x))
        target_x = target_x[indices]
        target_y = target_y[indices]
        
        # Iterate over mini-batches.
        for i in range(0, len(x), batch_size):
            end_i = i + batch_size
            batch_target_x = target_x[i:end_i]
            batch_target_y = target_y[i:end_i]
            current_batch_size = batch_target_x.shape[0]
            
            # Sample E1 and E2 for the current batch.
            E1 = torch.randn(current_batch_size, d1)
            E2 = torch.randn(current_batch_size, d2)
            
            # Forward pass.
            pred_X, pred_Y = model(E1, E2)
            pred_concat = torch.cat([pred_X, pred_Y], dim=1)
            batch_target_concat = torch.cat([batch_target_x, batch_target_y], dim=1)
            
            # Compute the MMD loss.
            loss = mmd_loss(pred_concat, batch_target_concat)
            
            # Backpropagation.
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1
        
        avg_loss = epoch_loss / num_batches if num_batches > 0 else float('inf')
        #print(f"Epoch [{epoch+1}], Average MMD Loss: {avg_loss:.4f}")
    return avg_loss

def cgnn(d,source="cdt",h=10,batch_size=128,epochs=30):
    x,y=d
    scaler = MinMaxScaler(feature_range=(-1,1))
    x = scaler.fit_transform(x)
    y = scaler.fit_transform(y)
    if source=="tiago":
        return -cgnn_score_tiago(x,y,h=h,batch_size=batch_size,num_epochs=epochs)+cgnn_score_tiago(y,x,h=h,batch_size=batch_size,num_epochs=epochs)
    else:
        return -cgnn_score(x,y,h=h,batch_size=batch_size,num_epochs=epochs)+cgnn_score(y,x,h=h,batch_size=batch_size,num_epochs=epochs)
   

In [8]:
print(ce.test_tuebingen(cgnn))
#print(ce.test_tuebingen(cgnn,source="tiago"))

(np.float64(0.6625114150297617), np.float64(0.6086532864590142))
